# Modis Water Functions

Notes:

- 8/12/2024 The GPU imports are not working (see below) 

In [1]:
from sklearn.ensemble import RandomForestClassifier as skRF
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.cluster import KMeans
from pathlib import Path   
import seaborn as sns
import pandas as pd
import numpy as np
import datetime
import warnings
import joblib
import pickle
import optuna
import time
import glob
import csv
import os

# # GPU-based frameworks
# from cuml.ensemble import RandomForestClassifier as cuRFC
# import cudf
# import cupy as cp

## Load Data

In [7]:
def CPU_Load_Data(fpath, colsToDrop, yCol='water', testSize=0.2, randomState=42, 
            dataType=np.int16, cpu=True, splitXY=False, trainTestSplit=False,
            applyLog=False, imbalance=False, frac=0.1, land=False, multi=False, 
            multisample=1000000):
    """
    Simple helper function for loading data to be used by models
    :param fpath: Path to the data to be ingested.
    :param dataType: Data type to convert ingested data to.
    :param colsToDrop: Columns which are not necessary, from which to drop.
    :param testSize: Ration to
    """
    if multi:
        all_dfs = [pd.read_csv(path_) for path_ in fpath]
        df = pd.concat(all_dfs).sample(n=multisample, random_state=randomState)
        print('DF length: {}'.format(len(df.index)))
    else:   
        df = pd.read_parquet(fpath) if '.parquet' in fpath else pd.read_csv(fpath)
    df = df[df['sur_refl_b01_1'] + df['sur_refl_b02_1'] != 0]
    df = df[df['sur_refl_b07_1'] + df['sur_refl_b02_1'] != 0]
    df = df[df['sur_refl_b06_1'] + df['sur_refl_b02_1'] != 0]

    df = df.drop(columns=colsToDrop)
    cleanedDF = df[~df.isin([np.NaN, np.inf, -np.inf]).any(1)].dropna(axis=0).astype(dataType)
    if applyLog:
        for col in cleanedDF.drop([yCol], axis=1).columns:
            print('Applying log1p func to {}'.format(col))
            cleanedDF[col] = np.log1p(cleanedDF[col])
        cleanedDF = cleanedDF[~cleanedDF.isin([np.NaN, np.inf, -np.inf]).any(1)].dropna(axis=0)
    df = None
    if imbalance:
        if land:
            print('Imbalancing data, sampling {} from water'.format(frac))
        else:
            print(f'Imbalancing data, sampling {frac} from land, {1-frac} from water')
        groupedDF = cleanedDF.groupby('water')
        dfs = [groupedDF.get_group(y) for y in groupedDF.groups]
        sampledDF = dfs[1].sample(frac=frac)if land else dfs[0].sample(frac=frac)
        concatDF = sampledDF.append(dfs[0]) if land else sampledDF.append(dfs[1])
        concatDF = concatDF.sample(frac=1)
        concatDF = concatDF.reset_index()
        cleanedDF = concatDF.drop(columns=['index'])
    if not splitXY:
        return cleanedDF
    X = cleanedDF.drop([yCol], axis=1).astype(dataType)
    y = cleanedDF[yCol].astype(dataType)
    if trainTestSplit:
        return train_test_split(X, y, test_size=TEST_RATIO)
    else:
        return X, y

In [ ]:
# def GPU_Load_Data(fpath, colsToDrop, yCol='water', testSize=0.2, randomState=42, 
#             dataType=cp.float32, cpu=False, splitXY=True, trainTestSplit=True,
#             applyLog=False, imbalance=False, frac=0.1, land=False, multi=False, 
#             multisample=1000000):
#     """
#     Simple helper function for loading data to be used by models
#     :param fpath: Path to the data to be ingested.
#     :param dataType: Data type to convert ingested data to.
#     :param colsToDrop: Columns which are not necessary, from which to drop.
#     :param testSize: Ration to
#     """
#     if multi:
#         all_dfs = [pd.read_csv(path_) for path_ in fpath]
#         df = pd.concat(all_dfs).sample(n=multisample, random_state=randomState)
#         print('DF length: {}'.format(len(df.index)))
#     else:   
#         df = pd.read_parquet(fpath) if '.parquet' in fpath else pd.read_csv(fpath)
#     df = df[df['sur_refl_b01_1'] + df['sur_refl_b02_1'] != 0]
#     df = df[df['sur_refl_b07_1'] + df['sur_refl_b02_1'] != 0]
#     df = df[df['sur_refl_b06_1'] + df['sur_refl_b02_1'] != 0]
#     df = df.drop(columns=colsToDrop)
#     cleanedDF = df[~df.isin([np.NaN, np.inf, -np.inf]).any(1)].dropna(axis=0).astype(dataType)
#     cleanedDF = cudf.from_pandas(cleanedDF) if not cpu else cleanedDF
#     if applyLog:
#         for col in cleanedDF.drop([yCol], axis=1).columns:
#             print('Applying log1p func to {}'.format(col))
#             cleanedDF[col] = np.log1p(cleanedDF[col])
#         cleanedDF = cleanedDF[~cleanedDF.isin([np.NaN, np.inf, -np.inf]).any(1)].dropna(axis=0)
#     df = None
#     if imbalance:
#         if land:
#             print('Imbalancing data, sampling {} from water'.format(frac))
#         else:
#             print('Imbalancing data, sampling {} from land'.format(frac))
#         groupedDF = cleanedDF.groupby('water')
#         dfs = [groupedDF.get_group(y) for y in groupedDF.groups]
#         sampledDF = dfs[1].sample(frac=frac)if land else dfs[0].sample(frac=frac)
#         concatDF = sampledDF.append(dfs[0]) if land else sampledDF.append(dfs[1])
#         concatDF = concatDF.sample(frac=1)
#         concatDF = concatDF.reset_index()
#         cleanedDF = concatDF.drop(columns=['index'])
#     if not splitXY:
#         return cleanedDF
#     X = cleanedDF.drop([yCol], axis=1).astype(dataType)
#     y = cleanedDF[yCol].astype(dataType)
#     cleanedX = cleanedDF.drop([yCol], axis=1).astype(dataType)
#     cleanedy = cleanedDF[yCol].astype(dataType)
#     if trainTestSplit:
#         return train_test_split(cleanedX, cleanedy, test_size=TEST_RATIO)
#     else:
#         return cleanedX, cleanedy

In [3]:
def Cluster_Counts(kmeans_output):
    '''
    In:
        kmeans_output (list): Kmeans output of the clusters assigned to each given datapoint
    Out:
       min_count (int): The minimum number of datapoints across the different clusters
    '''
    unique, counts = np.unique(kmeans_output, return_counts=True)
    min_count = np.nanmin(counts)
    print(min_count)
    return min_count

def Stratified_Cluster_Sampling(min_count,kmeans_output):
    '''
    In:
        min_count (int): The minimum number of datapoints desired
        kmeans_output (list): Kmeans output of the clusters assigned to each given datapoint
    Out:
       stratified_clusters (Array): Clusters with the same chosen min_count of datapoints
    '''
    stratified_clusters = np.array([])
    for cluster in np.unique(kmeans_output):
        print(f'Cluster {cluster}')  
        cluster_data = np.where(kmeans_output == cluster)[0]
        sample_data = np.random.choice(cluster_data,min_count,replace=False)
        stratified_clusters = np.append(stratified_clusters, sample_data)
    stratified_clusters = stratified_clusters.astype('int')
    print(len(stratified_clusters))
    print()
    return stratified_clusters

## Random forest

In [4]:
rf_search_space={
    "n_estimators": [75, 100, 125, 150, 175, 200, 250, 300, 400, 500],
    "max_depth" : [5,10, 30, 50, 80, 90, 100, 110],
    "min_samples_leaf" : [1, 2, 3, 4, 5],
    "min_samples_split" : [2, 4, 8, 10],
    "bootstrap" : [True, False],
    "max_features" : ['sqrt', 'log2'],
    
}

list_trees = [75, 100, 125, 150, 175, 200, 250, 300, 400, 500]
max_depth = [5, 10, 30, 50, 80, 90, 100, 110]
min_samples_leaf = [1, 2, 3, 4, 5]
min_samples_split = [2, 4, 8, 10]
bootstrap = [True, False]
max_features = ['sqrt', 'log2']

TREES_AND_DEPTH_ONLY = False
GRID_SEARCH = True

In [5]:
def CPU_RF_Objective(trial, X, y):
    param = {'n_estimators': trial.suggest_categorical('n_estimators', list_trees), 
           'max_depth':trial.suggest_categorical('max_depth', max_depth), 
           'min_samples_split':trial.suggest_categorical('min_samples_split', min_samples_split), 
           'min_samples_leaf':trial.suggest_categorical('min_samples_leaf', min_samples_leaf), 
           'bootstrap': trial.suggest_categorical('bootstrap', bootstrap),
           'criterion':'gini', 
           #'min_weight_fraction_leaf': trial.suggest_float('min_weight_fraction_leaf', 1e-8, 1.0, log=True), 
           'max_features':trial.suggest_categorical('max_features', max_features), 
           'max_leaf_nodes':None, 
           'min_impurity_decrease':0.0, 
           'oob_score':False, 
           'n_jobs':-1, 
           # 'random_state':42, 
           'verbose':0, 
           'warm_start':False, 
           'class_weight':None, 
           'ccp_alpha':0.0, 
           'max_samples':None
        }
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = np.empty(5)
    
    for idx, (train_idx, val_idx) in enumerate(cv.split(X, y)):
        try:
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        except:
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]
        
        model = skRF(**param)
        model.fit(X_train, y_train)
        preds = model.predict(X_val)
        cv_scores[idx] = f1_score(y_val, preds, average='weighted') 
        #using weighted for the multiclass because each class is balanced but shuffled
        
        if cv_scores[idx] == 0.0:
            print('Pruning because of 0.0 score.')
            return 0.0
        print('Fold {}: {}'.format(idx, cv_scores[idx]))
    return np.mean(cv_scores)

In [ ]:
# def GPU_RF_Objective(trial, X, y):
#     param = {'n_estimators': trial.suggest_categorical('n_estimators', list_trees), 
#         'max_depth':trial.suggest_categorical('max_depth', max_depth), 
#         'min_samples_split':trial.suggest_categorical('min_samples_split', min_samples_split), 
#         'min_samples_leaf':trial.suggest_categorical('min_samples_leaf', min_samples_leaf), 
#         'max_features':trial.suggest_categorical('max_features', max_features), 
#             }
#     cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
#     cv_scores = np.empty(5)
#     for idx, (train_idx, val_idx) in enumerate(cv.split(X.to_pandas(),y.to_pandas())):
#         X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
#         y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
#         model = cuRFC(**param)
#         model.fit(X_train, y_train)
#         preds = model.predict(X_val)
#         cv_scores[idx] = f1_score(y_val.to_numpy(), preds.to_numpy())
#         del model, preds
#         if cv_scores[idx] == 0.0:
#             print('Pruning because of 0.0 score.')
#             return 0.0
#         print('Fold {}: {}'.format(idx, cv_scores[idx]))
#     return np.mean(cv_scores)

In [6]:
def Tuning(obj, X_chosen, y_chosen, 
           ml_model = 'CPU', search_space = rf_search_space, Training = False,
           out_file = f'rfa_models/MODIS_RFA_v201_Cluster_no-outlier-cluster',
           save_pkl = False, NN = False, NTRIALS = 2):
    
    start_time = time.time()
    optuna.logging.set_verbosity(optuna.logging.INFO)
    if GRID_SEARCH:
        study = optuna.create_study(study_name='Tuning Grid Search', 
                                    direction='maximize',
                                    sampler=optuna.samplers.GridSampler(search_space))
    else:
        study = optuna.create_study(study_name='RF Tuning',
                                    direction='maximize')
        
    study.optimize(lambda trial: obj(trial, X_chosen, y_chosen), n_trials=NTRIALS, timeout=30*600)
    print("--- %s seconds ---" % (time.time() - start_time))
   
    #############################
    trials = study.best_trials            
    max_trial_score = max([trial.values[0] for trial in trials])
    max_trial_params = [trial.params for trial in trials 
                        if trial.values[0] == max_trial_score][0]
    max_trial_params['n_jobs'] = -1
    score_print = int(np.round(max_trial_score,4)*1000)
    print('\nMax score:', score_print)
    score_out_file = f'{out_file}_MaxScore{score_print}.pkl'
    print(score_out_file)
    
    #############################
    if Training: 
        start_time = time.time()
        hyperparameters = max_trial_params
        hyperparameters['n_jobs'] = -1
        print('Using these params:')
        print(hyperparameters)
        if ml_model =='CPU':
            tuned_classifier = skRF(**hyperparameters)
        elif ml_model == 'GPU':
            tuned_classifier = cuRFC(**hyperparameters)
        tuned_classifier.fit(X_chosen, y_chosen)
        if save_pkl: 
            pickle.dump(tuned_classifier, open(score_out_file, 'wb'))
        print("--- %s seconds ---" % (time.time() - start_time))